In [1]:
# 导入所需的库
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.notebook import tqdm # 提供进度条显示

# 导入 scTenifold 官方包
from scTenifold.core._networks import manifold_alignment, d_regulation

# 忽略绘图警告
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------
# 1. 全局路径与参数配置
# ----------------------------------------------------
# 输入文件路径 
EDGES_FILE = "pruned_edges.csv"

# 结果保存的主目录
OUTPUT_DIR = "Sctenifoldpy_RBP_KO_Analysis_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 流形对齐参数 
MANIFOLD_DIM = 15

# 设置随机种子保证可重复性
np.random.seed(42)

print(f"✅ 环境配置完毕。结果将保存在目录: {OUTPUT_DIR}/")

✅ 环境配置完毕。结果将保存在目录: Sctenifoldpy_RBP_KO_Analysis_Results/


In [2]:
# ----------------------------------------------------
# 2. 读取预计算权重，构建全局 WT 邻接矩阵
# ----------------------------------------------------
print("正在读取网络边权重...")
edges_df = pd.read_csv(EDGES_FILE)

# 提取所有的 RBP 和 Target 基因，形成全局节点集合
all_genes = list(set(edges_df['RBP']).union(set(edges_df['TG'])))
rbp_list = edges_df['RBP'].unique()

print(f"共发现 {len(rbp_list)} 个待测试的 RBPs。")
print(f"全局网络共包含 {len(all_genes)} 个节点 (RBP + Targets)。")

# 初始化全 0 方阵
WT_df = pd.DataFrame(0.0, index=all_genes, columns=all_genes)

# 使用 final_score 填充邻接矩阵 (行代表出度，列代表入度)
for _, row in edges_df.iterrows():
    WT_df.at[row['RBP'], row['TG']] = row['final_score']

# 将对角线置 0 (移除自调控边)
np.fill_diagonal(WT_df.values, 0.0)

print("✅ 全局 WT 邻接矩阵构建完成！")
# 可以在这预览一下矩阵大小
print(f"WT 矩阵维度: {WT_df.shape}")

正在读取网络边权重...
共发现 556 个待测试的 RBPs。
全局网络共包含 2270 个节点 (RBP + Targets)。
✅ 全局 WT 邻接矩阵构建完成！
WT 矩阵维度: (2270, 2270)


In [3]:
# ----------------------------------------------------
# 3. 批量虚拟敲除实验
# ----------------------------------------------------
all_target_results = []
# 用于存储具有显著靶基因的 RBP 及其显著靶标列表，方便后续画图
significant_rbp_dict = {} 

print("开始批量进行虚拟敲除与流形对齐 (DR Test)...")

# 使用 tqdm 显示进度条
for rbp in tqdm(rbp_list, desc="Processing RBPs"):
    # 1. 虚拟敲除：断开当前 RBP 的所有出向边
    KO_df = WT_df.copy()
    KO_df.loc[rbp, :] = 0.0
    
    # 2. 运行全局流形对齐
    ma_df = manifold_alignment(WT_df, KO_df, d=MANIFOLD_DIM)
    
    # 3. 差异调控检验
    dr_res = d_regulation(ma_df).set_index('Gene')
    
    # 4. 提取当前 RBP 的靶基因结果
    current_targets = edges_df[edges_df['RBP'] == rbp]['TG'].unique()
    target_dr = dr_res.loc[current_targets].copy()
    
    # 添加上下文信息
    target_dr['KO_RBP'] = rbp
    target_dr['Target_Gene'] = target_dr.index
    target_dr['is_significant'] = target_dr['adjusted p-value'] < 0.05
    target_dr['original_edge_weight'] = WT_df.loc[rbp, current_targets].values
    
    all_target_results.append(target_dr)
    
    # 如果该 RBP 有显著响应的靶基因，记录下来以备后续画图
    sig_targets = target_dr[target_dr['is_significant']]['Target_Gene'].tolist()
    if len(sig_targets) > 0:
        significant_rbp_dict[rbp] = sig_targets

print("✅ 所有 RBP 虚拟敲除实验完成！")

开始批量进行虚拟敲除与流形对齐 (DR Test)...


Processing RBPs:   0%|          | 0/556 [00:00<?, ?it/s]

manifold_alignment  processing time:  4.270642034010962
d_regulation  processing time:  0.6211025549564511
manifold_alignment  processing time:  5.883706594002433
d_regulation  processing time:  0.543688133941032
manifold_alignment  processing time:  5.002706996980123
d_regulation  processing time:  0.5434203899931163
manifold_alignment  processing time:  5.619482735055499
d_regulation  processing time:  0.5449693610426039
manifold_alignment  processing time:  6.04645080701448
d_regulation  processing time:  0.5427050000289455
manifold_alignment  processing time:  5.6841168000828475
d_regulation  processing time:  0.5422623560298234
manifold_alignment  processing time:  5.694398425985128
d_regulation  processing time:  0.5436857760651037
manifold_alignment  processing time:  5.015413827029988
d_regulation  processing time:  0.543232809053734
manifold_alignment  processing time:  5.622374758007936
d_regulation  processing time:  0.5419062849832699
manifold_alignment  processing time:  4

In [4]:
# ----------------------------------------------------
# 4. 汇总总表并筛选全局显著结果
# ----------------------------------------------------
final_results = pd.concat(all_target_results, ignore_index=True)

# 调整列的顺序
cols = ['KO_RBP', 'Target_Gene', 'original_edge_weight', 
        'Distance', 'FC', 'p-value', 'adjusted p-value', 'is_significant']
final_results = final_results[cols]

# 排序逻辑: 先按 RBP 分组，组内按显著性 p-value 升序
final_results = final_results.sort_values(by=['KO_RBP', 'adjusted p-value'])

# 1. 保存包含所有测试对的完整表格
full_report_path = os.path.join(OUTPUT_DIR, "All_Tested_RBP_Targets_Report.csv")
final_results.to_csv(full_report_path, index=False)

# 2. 保存只包含显著扰动靶标的精简表格 
sig_results = final_results[final_results['is_significant']]
sig_report_path = os.path.join(OUTPUT_DIR, "Significant_Only_Targets_Report.csv")
sig_results.to_csv(sig_report_path, index=False)

print(f"✅ 结果汇总完毕！")
print(f"  完整报告: {full_report_path}")
print(f"  显著结果精简版: {sig_report_path}")
print(f"\n【统计】：共测试了 {len(final_results)} 对 RBP-Target 关系，其中 {len(sig_results)} 对呈现显著扰动 (FDR < 0.05)。")
print(f"【发现】：共有 {len(significant_rbp_dict)} 个 RBP 在敲除后诱发了至少 1 个靶基因的显著变化。")

✅ 结果汇总完毕！
  完整报告: Sctenifoldpy_RBP_KO_Analysis_Results/All_Tested_RBP_Targets_Report.csv
  显著结果精简版: Sctenifoldpy_RBP_KO_Analysis_Results/Significant_Only_Targets_Report.csv

【统计】：共测试了 55533 对 RBP-Target 关系，其中 1412 对呈现显著扰动 (FDR < 0.05)。
【发现】：共有 551 个 RBP 在敲除后诱发了至少 1 个靶基因的显著变化。


In [5]:
# ----------------------------------------------------
# 5. 可视化绘图：Ego-Network (自我中心网络图) 与 Lollipop (棒棒糖图)
# ----------------------------------------------------
import networkx as nx
import seaborn as sns

def plot_ego_network(rbp_name, target_df, save_path):
    """绘制自我中心网络图：线宽代表原始权重，节点颜色代表显著性，大小代表扰动距离"""
    G = nx.DiGraph()
    
    # 添加中心节点 (RBP)
    G.add_node(rbp_name, size=800, color='#87CEFA', label=rbp_name)
    
    # 提取网络信息
    sizes = [800]
    colors = ['#87CEFA']
    labels = {rbp_name: rbp_name}
    
    for _, row in target_df.iterrows():
        tg = row['Target_Gene']
        dist = row['Distance']
        is_sig = row['is_significant']
        weight = row['original_edge_weight']
        
        # 节点大小映射扰动距离 (基础大小200 + 距离放大)，颜色映射是否显著
        node_size = 200 + (dist * 150)
        node_color = '#FF6347' if is_sig else '#D3D3D3' # 显著为番茄红，不显著为浅灰
        
        G.add_node(tg)
        G.add_edge(rbp_name, tg, weight=weight)
        
        sizes.append(node_size)
        colors.append(node_color)
        labels[tg] = tg

    fig, ax = plt.subplots(figsize=(8, 8))
    # 使用 spring_layout 让节点自动散开
    pos = nx.spring_layout(G, k=0.8, seed=42)
    
    # 提取边权重用于画线粗细
    edge_weights = [G[u][v]['weight'] * 15 for u, v in G.edges()] # 放大权重方便肉眼观察
    
    # 绘制节点和边
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=sizes, node_color=colors, alpha=0.9, edgecolors='white', linewidths=1.5)
    nx.draw_networkx_edges(G, pos, ax=ax, width=edge_weights, edge_color='gray', arrows=True, arrowsize=15, alpha=0.6)
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=9, font_weight='bold')
    
    # 增加图例
    import matplotlib.lines as mlines
    sig_patch = mlines.Line2D([], [], color='w', marker='o', markerfacecolor='#FF6347', markersize=10, label='Significant (FDR < 0.05)')
    nsig_patch = mlines.Line2D([], [], color='w', marker='o', markerfacecolor='#D3D3D3', markersize=10, label='Not Significant')
    ax.legend(handles=[sig_patch, nsig_patch], loc='upper right', fontsize=9)
    
    ax.set_title(f"Target Perturbation Network after {rbp_name} KO\n(Node Size ~ Perturbation Distance, Edge Width ~ Original Weight)", fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def plot_lollipop(rbp_name, target_df, save_path):
    """绘制棒棒糖图：直观展示靶基因扰动距离排序"""
    # 按照 Distance (扰动程度) 排序
    df_sorted = target_df.sort_values(by='Distance', ascending=True).tail(30) # 如果靶基因太多，只画受影响最大的 Top 30
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # 区分颜色
    colors = ['#FF6347' if sig else '#D3D3D3' for sig in df_sorted['is_significant']]
    
    # 画棒棒糖的棍子
    ax.hlines(y=df_sorted['Target_Gene'], xmin=0, xmax=df_sorted['Distance'], color=colors, alpha=0.7, linewidth=2)
    # 画棒棒糖的糖（散点）
    ax.scatter(df_sorted['Distance'], df_sorted['Target_Gene'], color=colors, s=80, alpha=1, zorder=3, edgecolors='white')
    
    ax.set_xlabel('Perturbation Distance (Larger means more affected)', fontsize=10)
    ax.set_ylabel('Target Genes', fontsize=10)
    ax.set_title(f"Top Affected Targets of {rbp_name} Knockout", fontsize=12)
    
    # 优化网格
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


# ----------------------------------------------------
# 批量执行画图
# ----------------------------------------------------
print("正在为产生显著改变的 RBP 绘制 直观网络图 和 棒棒糖图...")

for rbp in tqdm(significant_rbp_dict.keys(), desc="Plotting Intuitive Figures"):
    # 创建文件夹
    rbp_dir = os.path.join(OUTPUT_DIR, f"KO_{rbp}")
    os.makedirs(rbp_dir, exist_ok=True)
    
    # 提取该 RBP 的所有靶基因结果
    target_df = final_results[final_results['KO_RBP'] == rbp]
    
    # 1. 保存网络图
    net_path = os.path.join(rbp_dir, f"EgoNetwork_{rbp}.png")
    plot_ego_network(rbp, target_df, net_path)
    
    # 2. 保存棒棒糖图
    lolli_path = os.path.join(rbp_dir, f"Lollipop_{rbp}.png")
    plot_lollipop(rbp, target_df, lolli_path)

print(f"✅ 直观可视化绘图完成！请前往 {OUTPUT_DIR}/ 目录下查看网络图和棒棒糖图。")

正在为产生显著改变的 RBP 绘制 直观网络图 和 棒棒糖图...


Plotting Intuitive Figures:   0%|          | 0/551 [00:00<?, ?it/s]

✅ 直观可视化绘图完成！请前往 Sctenifoldpy_RBP_KO_Analysis_Results/ 目录下查看网络图和棒棒糖图。
